In [43]:
import os
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from sklearn import metrics
import logging
import os
import sys
from cgi import test
from glob import glob
from pydoc import describe
import numpy as np
import pandas as pd
from joblib import dump, load
from matplotlib import pyplot as plt
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import (
    BayesianRidge,
    ElasticNet,
    LinearRegression,
    SGDRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import RegressorChain
# from sklearnex import patch_sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from math import sqrt
from numpy import concatenate
from matplotlib import pyplot
from pandas import read_csv
from pandas import DataFrame
from pandas import concat
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Input, Dense, LSTM, Conv1D, Dropout, Bidirectional, Multiply, MaxPooling1D, BatchNormalization
from tensorflow.keras.layers import InputLayer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [44]:
# convert series to supervised learning
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = DataFrame(data)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    # put it all together
    agg = concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [45]:

# Đọc và tiền xử lý dữ liệu
df_pre = pd.read_csv('Merged_PM2.5_AQI_18_23.csv', sep=",", dtype=str)
df_pre = df_pre[['AQI', 'wd', 'ws', 'Rainfall', 'low_leaf', 'temp', 'uvb']]
values = df_pre.values.astype('float32')


In [46]:
values = df_pre.values
# ensure all data is float
values = values.astype('float32')
temp = pd.DataFrame(values)
temp.head()

,0,1,2,3,4,5,6
0,66.0,353.0,1.4,0.0,1.923742,297.396729,0.0
1,75.0,349.0,1.3,0.0,1.923694,297.017334,0.0
2,85.0,346.0,1.4,0.0,1.923688,296.338867,0.0
3,77.0,330.0,1.4,0.0,1.923697,295.851562,0.0
4,93.0,350.0,1.4,0.0,1.923649,295.656494,0.0


In [47]:
# normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

In [48]:
#Single timestep no scale
# frame as supervised learning
n_hours = 24 # check timestep change  timestep to evaluate performance


n_features = 7  # fixed 7 values in data  (PM2.5_s	wd_s	ws_s	Rainfall_s	lowleaf_s	temp_s	uvb_s)
reframed = series_to_supervised(scaled, n_hours, 1)  # apply scaled not values
reframed.head()

,var1(t-24),var2(t-24),var3(t-24),var4(t-24),var5(t-24),var6(t-24),var7(t-24),var1(t-23),var2(t-23),var3(t-23),...,var5(t-1),var6(t-1),var7(t-1),var1(t),var2(t),var3(t),var4(t),var5(t),var6(t),var7(t)
31,0.323194,0.130556,0.138889,0.0,0.483862,0.389717,0.339507,0.315589,0.125000,0.097222,...,0.482733,0.236444,0.182963,0.372624,0.175000,0.194444,0.0,0.482793,0.394567,0.386021
32,0.315589,0.125000,0.097222,0.0,0.483972,0.455334,0.561489,0.300380,0.005556,0.083333,...,0.482793,0.394567,0.386021,0.365019,0.297222,0.138889,0.0,0.482552,0.455812,0.575588
33,0.300380,0.005556,0.083333,0.0,0.483684,0.504055,0.719310,0.307985,0.869444,0.083333,...,0.482552,0.455812,0.575588,0.384030,0.422222,0.138889,0.0,0.482615,0.506620,0.705661
34,0.307985,0.869444,0.083333,0.0,0.483743,0.592289,0.770408,0.353612,0.861111,0.041667,...,0.482615,0.506620,0.705661,0.418251,0.469444,0.166667,0.0,0.482455,0.591457,0.780999
35,0.353612,0.861111,0.041667,0.0,0.483775,0.636950,0.780660,0.387833,0.469444,0.013889,...,0.482455,0.591457,0.780999,0.406844,0.466667,0.166667,0.0,0.482483,0.603487,0.753259


In [49]:
# split into train and test sets
values = reframed.values
n_train_hours = 32469 - n_hours -1  # het nam 2021
train = values[:n_train_hours, :]
test = values[n_train_hours:, :]   # test dau nam 2022 den 2023

In [50]:
# split into input and outputs
n_obs = n_hours * n_features
print(n_obs)
train_X, train_y = train[:, :n_obs], train[:, -n_features]
test_X, test_y = test[:, :n_obs], test[:, -n_features]
print(train_X.shape, len(train_X), train_y.shape)


168
(32444, 168) 32444 (32444,)


In [51]:
# split into train and test sets
values = reframed.values
n_train_hours = 32469 - n_hours - 1  # het nam 2021
train = values[:n_train_hours, :]
test = values[n_train_hours:, :]   # test dau nam 2022 den 2023


In [52]:
# split into input and outputs
n_obs = n_hours * n_features
print(n_obs)
train_X, train_y = train[:, :n_obs], train[:, -n_features]
test_X, test_y = test[:, :n_obs], test[:, -n_features]
print(train_X.shape, len(train_X), train_y.shape)

168
(32444, 168) 32444 (32444,)


In [53]:
# reshape input to be 3D [samples, timesteps, features]
train_X = train_X.reshape((train_X.shape[0], n_hours, n_features))
test_X = test_X.reshape((test_X.shape[0], n_hours, n_features))
print(train_X.shape, train_y.shape, test_X.shape, test_y.shape)

(32444, 24, 7) (32444,) (5162, 24, 7) (5162,)


In [54]:
print(train_X.shape[1])
print(train_X.shape[2])

24
7


In [55]:
def LSTMandCNN(input_shape):
    inputs = Input(shape=input_shape)

    # Convolution layers with Batch Normalization
    x = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(filters=128, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)

    # LSTM layers
    bi1 = Bidirectional(LSTM(64, activation='tanh', return_sequences=True))(x)
    bi2 = Bidirectional(LSTM(32, activation='tanh', return_sequences=False))(bi1)

    # Dropout to prevent overfitting
    dropout = Dropout(0.3)(bi2)

    # Output layer
    dense3 = Dense(1)(dropout)

    model = Model(inputs, dense3)
    return model

# Khởi tạo mô hình
model = LSTMandCNN(input_shape=(train_X.shape[1], train_X.shape[2]))
model.summary()

# Compile mô hình với optimizer đã điều chỉnh
optimizer = Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss='mse')
'''
# model 2
def LSTMandCNN(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=128, kernel_size=1, activation='relu')(inputs)
    bi1 = Bidirectional(LSTM(64, activation='relu', return_sequences=True))(x)
    max1d= tf.keras.layers.GlobalAveragePooling1D()(bi1)
    #dropout = Dropout(0.2)(max1d)
    #dense1=Dense(128)(dropout)
    #dense2=Dense(64)(dense1)
    dense3=Dense(1)(max1d)
    model=Model([inputs],dense3)
    return model

model = LSTMandCNN(input_shape=(train_X.shape[1], train_X.shape[2]))
model.summary()
model.compile(optimizer = 'adam',loss = 'mse')
'''

'''
Test RMSE: 3.5072
Test MAE: 2.5244
Test R2: 0.9203
'''

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 24, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 22, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 22, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 20, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 20, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 166,977 (652.25 KB)

 Trainable params: 166,593 (650.75 KB)

 Non-trainable params: 384 (1.50 KB)

'\nTest RMSE: 3.5072\nTest MAE: 2.5244\nTest R2: 0.9203\n'

In [ ]:
from keras.callbacks import EarlyStopping

history = model.fit(train_X, train_y,
                    epochs=67,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=30, verbose=1,
                               mode='min', restore_best_weights=True)])
# plot history
pyplot.plot(history.history['loss'], label='train')
pyplot.plot(history.history['val_loss'], label='test')
pyplot.legend()
pyplot.show()

Epoch 1/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 54s 55ms/step - loss: 0.0380 - val_loss: 0.0096
Epoch 2/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 48s 59ms/step - loss: 0.0107 - val_loss: 0.0036
Epoch 3/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 47s 58ms/step - loss: 0.0070 - val_loss: 0.0030
Epoch 4/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 44s 55ms/step - loss: 0.0052 - val_loss: 0.0022
Epoch 5/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 86s 59ms/step - loss: 0.0042 - val_loss: 0.0017
Epoch 6/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 78s 55ms/step - loss: 0.0035 - val_loss: 0.0019
Epoch 7/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - loss: 0.0031 - val_loss: 0.0013
Epoch 8/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 83s 56ms/step - loss: 0.0028 - val_loss: 0.0012
Epoch 9/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 45s 55ms/step - loss: 0.0025 - val_loss: 0.0010
Epoch 10/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 49s 60ms/step - loss: 0.0025 - val_loss: 0.0012
Epoch 11/67
812/812 ━━━━━━━━━━━━━━━━━━━━ 78s 55ms/step - loss: 0.0022 - val_loss: 0.0011
Epoch 12/67
812/812 ━━━━━━━━━━

In [ ]:
# make a prediction
yhat = model.predict(test_X)
test_X = test_X.reshape((test_X.shape[0], n_hours*n_features))
# invert scaling for forecast
inv_yhat = concatenate((yhat, test_X[:, -(n_features-1):]), axis=1)
inv_yhat = scaler.inverse_transform(inv_yhat)
inv_yhat = inv_yhat[:, 0]
# invert scaling for actual
test_y = test_y.reshape((len(test_y), 1))
inv_y = concatenate((test_y, test_X[:, -(n_features-1):]), axis=1)
inv_y = scaler.inverse_transform(inv_y)
inv_y = inv_y[:, 0]

In [ ]:
# calculate RMSE
rmse = sqrt(mean_squared_error(inv_y, inv_yhat))
print('Test RMSE: %.4f' % rmse)

#calculate MAE
mae = mean_absolute_error(inv_y, inv_yhat)
print('Test MAE: %.4f' % mae)

# r-squared score of the model
r2 = r2_score(inv_y, inv_yhat)
print('Test R2: %.4f'% r2)

In [ ]:
#Plot the graph between actual vs predicted values
plt.figure(figsize=(10,6))
plt.plot(inv_yhat[:500], color= 'blue',label = 'Predicted Pollution level')
plt.plot(inv_y[:500] , color = 'red',label = 'Actual Pollution level')
plt.title("Air Pollution Prediction (Multivariate)")
plt.xlabel("Date")
plt.ylabel("Pollution level")
plt.legend()
plt.show()
plt.savefig('graph.png')

In [ ]:
# Chuyển giá trị thực tế và giá trị dự đoán thành DataFrame
df_results = pd.DataFrame({'Actual': inv_y, 'Predicted': inv_yhat})
# Lưu DataFrame vào file CSV
file_name = f'prediction_results_AQI_LSTM_CNN_{n_hours}h.csv'
df_results.to_csv(file_name, index=False)